# 04 - Model Comparison 
### Week 5: 
- Decision Tree & Random Forest regressors, 
- compare their test R^2 against baseline
- document model behavior (strengths/weakness)

## Decision Tree

In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error

In [2]:
train_df = pd.read_csv("data/train_final.csv")
test_df = pd.read_csv("data/test_final.csv")

drop_from_features = ['ClosePrice', 'ClosePrice_log', 'CloseDate', 'CloseYearMonth']

X_train = train_df.drop(columns=[c for c in drop_from_features if c in train_df.columns])
X_test = test_df.drop(columns=[c for c in drop_from_features if c in test_df.columns])

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

y_train = train_df['ClosePrice_log']
y_test = test_df['ClosePrice_log']

In [3]:
dt_model = DecisionTreeRegressor(max_depth=10, random_state=42)
dt_model.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=10, random_state=42)

In [4]:
dt_pred_log = dt_model.predict(X_test)
dt_pred_price = np.exp(dt_pred_log)
actual_price = np.exp(y_test)

dt_r2 = r2_score(y_test, dt_pred_log)
dt_mape = mean_absolute_percentage_error(actual_price, dt_pred_price)
dt_mdape = np.median(np.abs((actual_price - dt_pred_price) / actual_price))

print(f"Decision Tree — R²: {dt_r2:.4f}, MAPE: {dt_mape:.4f}, MdAPE: {dt_mdape:.4f}")

Decision Tree — R²: 0.7640, MAPE: 0.2972, MdAPE: 0.1483


# Random Forest

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

In [ ]:
rf_pred_log = rf_model.predict(X_test)
rf_pred_price = np.exp(rf_pred_log)

rf_r2 = r2_score(y_test, rf_pred_log)
rf_mape = mean_absolute_percentage_error(actual_price, rf_pred_price)
rf_mdape = np.median(np.abs((actual_price - rf_pred_price) / actual_price))

print(f"Random Forest — R²: {rf_r2:.4f}, MAPE: {rf_mape:.4f}, MdAPE: {rf_mdape:.4f}")

Random Forest — R²: 0.8847, MAPE: 0.2343, MdAPE: 0.1036


In [ ]:
comparison = pd.DataFrame({
    'Model': ['Linear Regression (baseline)', 'Decision Tree', 'Random Forest'],
    'R2': [0.7716, dt_r2, rf_r2],
    'MAPE': [0.2738, dt_mape, rf_mape],
    'MdAPE': [0.1691, dt_mdape, rf_mdape]
})
print(comparison)

                          Model        R2      MAPE     MdAPE
0  Linear Regression (baseline)  0.771600  0.273800  0.169100
1                 Decision Tree  0.764036  0.297194  0.148317
2                 Random Forest  0.884650  0.234267  0.103589


# 04 - Model Comparison — Summary

**What this notebook does:**
1. Loads the same `train_final.csv` / `test_final.csv` and feature setup as `03_baseline_model.ipynb`
2. Trains a Decision Tree regressor (`max_depth=10`)
3. Trains a Random Forest regressor (`n_estimators=100, max_depth=15`)
4. Compares both against the Linear Regression baseline

**Results:**

| Model | R² | MAPE | MdAPE |
|---|---|---|---|
| Linear Regression | 0.7716 | 0.2738 | 0.1691 |
| Decision Tree | 0.7640 | 0.2972 | 0.1483 |
| **Random Forest** | **0.8847** | **0.2343** | **0.1036** |

**Takeaway:** Random Forest wins across all three metrics — explains ~88% of price variance and nearly halves median error vs. baseline. Averaging across 100 trees reduces the overfitting a single tree is prone to, while both tree models capture nonlinear relationships Linear Regression can't.

**Known limitations:**

- High-cardinality location columns (City, SubdivisionName, PostalCode, ElementarySchool, MLSAreaMajor, HighSchoolDistrict) were dropped to keep one-hot dimensionality manageable
- 29 rows missing Lat/Long were geocoded via Nominatim (16 recovered, 13 dropped as ungeocodable)
- Outlier cutoff used a $100M hard ceiling instead of the team's $5M cap / bottom-percentile trim discussion